In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
train_data = pd.read_csv(
    '/Users/KeanuVentura/Desktop/data/train.csv',
    dtype={'zipcode': str},
    low_memory=False
)

test_data = pd.read_csv(
    '/Users/KeanuVentura/Desktop/data/test.csv',
    dtype={'zipcode': str},
    low_memory=False
)

benchmark = pd.read_csv('mean_value_baseline.csv')

#train_data.head()
#test_data.head()
#benchmark.head()

print(train_data.shape, train_data.columns, train_data.info(), train_data.describe())

<class 'pandas.DataFrame'>
RangeIndex: 33538 entries, 0 to 33537
Data columns (total 65 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                33538 non-null  int64  
 1   name                              33526 non-null  str    
 2   summary                           32266 non-null  str    
 3   space                             23038 non-null  str    
 4   description                       33230 non-null  str    
 5   experiences_offered               33538 non-null  str    
 6   neighborhood_overview             19948 non-null  str    
 7   notes                             13445 non-null  str    
 8   transit                           20796 non-null  str    
 9   access                            19304 non-null  str    
 10  interaction                       18658 non-null  str    
 11  house_rules                       19983 non-null  str    
 12  host_id        

In [ ]:
X_test = test_data[features]

test_preds_log = model.predict(X_test)
test_preds = np.expm1(test_preds_log)

# clip negatives
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    'Id': test_data['id'],
    'Predicted': test_preds
})

submission.to_csv('submission_v2.csv', index=False)

In [3]:
num_cols = train_data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = train_data.select_dtypes(include=['object']).columns

print("Numeric:", len(num_cols))
print("Categorical/Text:", len(cat_cols))

missing = train_data.isnull().mean().sort_values(ascending=False)
missing.head(20)

Numeric: 23
Categorical/Text: 42


/var/folders/xw/_wbvd7ps4xv0mqqnxnwfzcb00000gn/T/ipykernel_75197/444883969.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_data.select_dtypes(include=['object']).columns


host_acceptance_rate           1.000000
square_feet                    0.989832
notes                          0.599111
host_response_rate             0.482825
host_response_time             0.482825
interaction                    0.443676
access                         0.424414
neighborhood_overview          0.405212
house_rules                    0.404168
host_about                     0.392510
transit                        0.379927
space                          0.313078
review_scores_value            0.229918
review_scores_checkin          0.229859
review_scores_location         0.229829
review_scores_accuracy         0.229411
review_scores_communication    0.229262
review_scores_cleanliness      0.228964
review_scores_rating           0.228517
reviews_per_month              0.207138
dtype: float64

In [4]:
features = [
    'accommodates',
    'bathrooms',
    'bedrooms',
    'beds',
    'room_type',
    'neighbourhood_group_cleansed',
    'number_of_reviews',
    'review_scores_rating',
    'reviews_per_month',
    'host_is_superhost'
]

num_cols = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month'
]

for col in num_cols:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    test_data[col] = test_data[col].fillna(train_data[col].median())

cat_cols = ['room_type', 'neighbourhood_group_cleansed', 'host_is_superhost']

for col in cat_cols:
    train_data[col] = train_data[col].fillna('missing')
    test_data[col] = test_data[col].fillna('missing')

In [5]:
X = train_data[features]
y = np.log1p(train_data['price'])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month'
]

categorical_features = [
    'room_type', 'neighbourhood_group_cleansed', 'host_is_superhost'
]

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [6]:
val_preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))

print("Validation RMSE:", rmse)

Validation RMSE: 0.43837259493603226


In [7]:
X_test = test_data[features]

test_preds_log = model.predict(X_test)
test_preds = np.expm1(test_preds_log)

test_preds = np.maximum(0, test_preds)
submission = pd.DataFrame({
    'Id': test_data['id'],
    'Predicted': test_preds
})

submission.to_csv('submission_v1.csv', index=False)

In [8]:
# attempt 2

In [9]:
train_data['host_response_rate'] = (
    train_data['host_response_rate']
    .astype(str)
    .str.replace('%', '', regex=False)
    .replace('nan', np.nan)
    .astype(float)
)

test_data['host_response_rate'] = (
    test_data['host_response_rate']
    .astype(str)
    .str.replace('%', '', regex=False)
    .replace('nan', np.nan)
    .astype(float)
)

train_data['extra_people'] = (
    train_data['extra_people']
    .astype(str)
    .str.replace('$', '', regex=False)
    .replace('nan', np.nan)
    .astype(float)
)

test_data['extra_people'] = (
    test_data['extra_people']
    .astype(str)
    .str.replace('$', '', regex=False)
    .replace('nan', np.nan)
    .astype(float)
)

In [11]:
features = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',

    'room_type', 'neighbourhood_group_cleansed',

    'number_of_reviews', 'review_scores_rating', 'reviews_per_month',

    'host_is_superhost', 'host_response_rate',

    'extra_people', 'minimum_nights'
]

num_cols = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month',
    'host_response_rate', 'extra_people',
    'minimum_nights'
]

cat_cols = [
    'room_type', 'neighbourhood_group_cleansed', 'host_is_superhost'
]

for col in num_cols:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    test_data[col] = test_data[col].fillna(train_data[col].median())

for col in cat_cols:
    train_data[col] = train_data[col].fillna('missing')
    test_data[col] = test_data[col].fillna('missing')

In [16]:
X = train_data[features]
y = np.log1p(train_data['price'])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [18]:
val_preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))

print("Validation RMSE:", rmse)

Validation RMSE: 0.40008220929660704


In [19]:
X_test = test_data[features]

test_preds_log = model.predict(X_test)
test_preds = np.expm1(test_preds_log)

# clip negatives
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    'Id': test_data['id'],
    'Predicted': test_preds
})

submission.to_csv('submission_v2.csv', index=False)

In [ ]:
# attempt 3

In [20]:
def parse_amenities(x):
    if pd.isna(x):
        return []
    return [a.strip().lower() for a in x.replace('{','').replace('}','').split(',')]

train_data['amenities_list'] = train_data['amenities'].apply(parse_amenities)
test_data['amenities_list'] = test_data['amenities'].apply(parse_amenities)

from collections import Counter

all_amenities = Counter()
for lst in train_data['amenities_list']:
    all_amenities.update(lst)

top_amenities = [a for a, _ in all_amenities.most_common(25)]

for amenity in top_amenities:
    train_data[f'amenity_{amenity}'] = train_data['amenities_list'].apply(lambda x: int(amenity in x))
    test_data[f'amenity_{amenity}'] = test_data['amenities_list'].apply(lambda x: int(amenity in x))

train_data['host_since'] = pd.to_datetime(train_data['host_since'], errors='coerce')
test_data['host_since'] = pd.to_datetime(test_data['host_since'], errors='coerce')

train_data['host_years'] = (pd.Timestamp('2020-01-01') - train_data['host_since']).dt.days / 365
test_data['host_years'] = (pd.Timestamp('2020-01-01') - test_data['host_since']).dt.days / 365

train_data['host_years'] = train_data['host_years'].fillna(train_data['host_years'].median())
test_data['host_years'] = test_data['host_years'].fillna(train_data['host_years'].median())

train_data['has_reviews'] = (train_data['number_of_reviews'] > 0).astype(int)
test_data['has_reviews'] = (test_data['number_of_reviews'] > 0).astype(int)

base_features = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'room_type', 'neighbourhood_group_cleansed',
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month',
    'host_is_superhost', 'host_response_rate',
    'extra_people', 'minimum_nights',
    'host_years', 'has_reviews'
]

amenity_features = [f'amenity_{a}' for a in top_amenities]

features = base_features + amenity_features

num_cols = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month',
    'host_response_rate', 'extra_people', 'minimum_nights',
    'host_years', 'has_reviews'
] + amenity_features

cat_cols = ['room_type', 'neighbourhood_group_cleansed', 'host_is_superhost']

for col in num_cols:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    test_data[col] = test_data[col].fillna(train_data[col].median())

for col in cat_cols:
    train_data[col] = train_data[col].fillna('missing')
    test_data[col] = test_data[col].fillna('missing')

preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42,
        n_jobs=-1
    ))
])

X = train_data[features]
y = np.log1p(train_data['price'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)

val_preds = model.predict(X_val)


X_test = test_data[features]

test_preds = np.expm1(model.predict(X_test))
test_preds = np.maximum(0, test_preds)

In [23]:
rmse = np.sqrt(mean_squared_error(y_val, val_preds))

print("New Validation RMSE:", rmse)

New Validation RMSE: 0.38206873350266846


In [22]:
submission = pd.DataFrame({
    'Id': test_data['id'],
    'Predicted': test_preds
})

submission.to_csv('submission_v3.csv', index=False)



In [ ]:
# attempt 4

In [27]:
train_data['text'] = (
    train_data['description'].fillna('') + ' ' +
    train_data['summary'].fillna('')
)

test_data['text'] = (
    test_data['description'].fillna('') + ' ' +
    test_data['summary'].fillna('')
)

tfidf = TfidfVectorizer(
    max_features=300,
    stop_words='english'
)

preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('text', tfidf, 'text')
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=700,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42,
        n_jobs=-1
    ))
])

X = train_data[features + ['text']]
y = np.log1p(train_data['price'])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

val_preds = model.predict(X_val)

In [28]:
rmse = np.sqrt(mean_squared_error(y_val, val_preds))

print("FINAL RMSE:", rmse)

FINAL RMSE: 0.3658473642370436


In [29]:
X_test = test_data[features + ['text']]

test_preds = np.expm1(model.predict(X_test))
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    'Id': test_data['id'],
    'Predicted': test_preds
})

submission.to_csv('submission_v4.csv', index=False)